<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp2_a_finbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Thesis**: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

**Component: Experiment 2** - Model A: FinBERT (ProsusAI/finbert)

**Description:** This notebook loads the prepared 1000 headline dataset and evaluates FinBERT on Experiment 2. It classifies each headline as positive or negative, maps predictions to directional signals, and computes directional accuracy, precision, recall and F1-score based on next-day stock price movement.

Select T4 GPU as runtime.

In [ ]:
# Install required libraries — Experiment 3a: FinBERT

!pip install -q transformers
!pip install -q pandas scikit-learn matplotlib seaborn

print("Libraries installed successfully.")

In [ ]:
# Import required libraries — Experiment 2a: FinBERT

import pandas as pd
import numpy as np
import torch
import warnings

from transformers import pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face and load Experiment 2 dataset

from google.colab import userdata, drive
from huggingface_hub import login

drive.mount("/content/drive")

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

df_exp2 = pd.read_csv("/content/drive/MyDrive/Thesis_Data/exp2_dataset.csv")

print("Dataset loaded.")
print(f"Total headlines : {len(df_exp2)}")
print(f"Columns : {list(df_exp2.columns)}")
print(f"\nMovement distribution:")
print(df_exp2["movement"].value_counts())

In [ ]:
# Load FinBERT

print("Loading FinBERT...")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1
)

print("FinBERT loaded successfully.")

In [ ]:
# Run FinBERT on 1,000 headlines

headlines = df_exp2["headline"].tolist()

print("Running FinBERT on 1,000 headlines...")

finbert_results = finbert(headlines, batch_size=32)
finbert_preds = [r["label"].lower() for r in finbert_results]

df_exp2["finbert_sentiment"] = finbert_preds

print(f"Done. Total predictions: {len(finbert_preds)}")
print(f"\nSentiment distribution:")
print(pd.Series(finbert_preds).value_counts())

In [ ]:
# Map sentiment to directional prediction and evaluate
# Positive → 1 (price goes up)
# Negative → 0 (price goes down)

df_exp2["finbert_direction"] = df_exp2["finbert_sentiment"].map({
    "positive": 1,
    "negative": 0
})

true_movement = df_exp2["movement"].tolist()
pred_movement = df_exp2["finbert_direction"].tolist()

# Evaluate
acc  = accuracy_score(true_movement, pred_movement)
prec = precision_score(true_movement, pred_movement, average="macro")
rec  = recall_score(true_movement, pred_movement, average="macro")
f1   = f1_score(true_movement, pred_movement, average="macro")

print("=" * 50)
print("FinBERT — Experiment 2 Results (next-day movement)")
print("=" * 50)
print(f"  Directional Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision            : {prec:.4f}")
print(f"  Recall               : {rec:.4f}")
print(f"  F1-Score             : {f1:.4f}")
print("=" * 50)

print("\nDetailed Classification Report:")
print(classification_report(true_movement, pred_movement,
      target_names=["Down (0)", "Up (1)"]))

In [ ]:
# Save FinBERT Experiment 2 results

finbert_exp2_scores = {
    "model": "FinBERT",
    "directional_accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4)
}

df_exp2["finbert_direction"] = pred_movement

print("FinBERT — Experiment 2 Results Summary")
print(pd.DataFrame([finbert_exp2_scores]))

# Save to Google Drive
from google.colab import drive
drive.mount("/content/drive")

df_exp2.to_csv("/content/drive/MyDrive/Thesis_Data/exp2_finbert_preds.csv", index=False)
print("\nSaved to Google Drive.")